# TP 7 / 10 — Modèles ML avancés : arbre, forêt, gradient boosting

**M2 Actuariat — Abidjan**

---

## Objectifs
- Sortir du cadre GLM et essayer des modèles **non linéaires** qui capturent automatiquement les interactions.
- Construire et **comparer** trois familles : **arbre de décision**, **forêt aléatoire (Random Forest)**, **gradient boosting (LightGBM)**.
- Mettre en place une **recherche d'hyperparamètres** sérieuse (`RandomizedSearchCV`).
- **Comparer rigoureusement** GLM amélioré (TP 5) vs ces nouveaux modèles, sur AUC, Gini et lift.
- Examiner les **importances de variables** d'un modèle de gradient boosting et les confronter aux coefficients GLM.

**Durée estimée : ~45 min.**

---


## 0. Imports et chargement


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve

import lightgbm as lgb

sns.set_theme(style="whitegrid")
np.random.seed(42)

X_train = pd.read_csv("X_train.csv").astype(float)
X_test  = pd.read_csv("X_test.csv").astype(float)
y_train = pd.read_csv("y_train.csv").squeeze("columns")
y_test  = pd.read_csv("y_test.csv").squeeze("columns")
metrics_baseline = pd.read_csv("metrics_glm_baseline.csv", index_col=0).squeeze("columns")

print("Shapes :", X_train.shape, X_test.shape)
print("Taux BAD train :", f"{y_train.mean():.3%}", "| test :", f"{y_test.mean():.3%}")


## 1. Arbre de décision

L'arbre est le modèle **le plus simple** des trois, et le plus **interprétable** : on peut le dessiner. Risque principal : **surapprentissage** si on le laisse pousser sans limite → on impose `max_depth`.


In [ ]:
tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=200, random_state=42)
tree.fit(X_train, y_train)

p_tree_test = tree.predict_proba(X_test)[:, 1]
auc_tree = roc_auc_score(y_test, p_tree_test)
print(f"AUC arbre (test) : {auc_tree:.4f}")
print(f"Profondeur effective : {tree.get_depth()}, feuilles : {tree.get_n_leaves()}")


In [ ]:
# Visualisation de l'arbre (limite à 3 niveaux pour la lisibilité)
fig, ax = plt.subplots(figsize=(20, 9))
plot_tree(
    tree, feature_names=X_train.columns.tolist(),
    class_names=["GOOD", "BAD"], filled=True, rounded=True, max_depth=3,
    fontsize=9, ax=ax
)
plt.title("Arbre de décision — 3 premiers niveaux")
plt.show()


### Question 1
- Quelle est la **toute première variable** utilisée par l'arbre pour split ? Est-ce cohérent avec les coefficients du GLM (TP 3) ?
- Si on retire la contrainte `min_samples_leaf=200`, à votre avis qu'arrive-t-il à l'AUC sur le **train** ? Sur le **test** ?
- Comment l'arbre représente-t-il une **interaction** entre deux variables, sans qu'on ait à la coder explicitement ?

*Votre réponse :*


## 2. Random Forest

Une **forêt** = plusieurs arbres, chacun entraîné sur un sous-échantillon bootstrap et un sous-ensemble de variables → moyenne des probas. Cela **réduit drastiquement la variance** d'un arbre isolé.


In [ ]:
rf = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=50,
    n_jobs=-1, random_state=42,
)
rf.fit(X_train, y_train)
p_rf_test = rf.predict_proba(X_test)[:, 1]
auc_rf = roc_auc_score(y_test, p_rf_test)
print(f"AUC Random Forest (test) : {auc_rf:.4f}")


### Question 2
- Quel paramètre joue le rôle de **régularisation** principal dans un Random Forest ? (Indice : `max_depth`, `min_samples_leaf`, `max_features`, `n_estimators` — lequel est le plus impactant ?)
- Pourquoi un Random Forest est-il **moins sujet au surapprentissage** qu'un arbre seul, **même** avec une grande profondeur ?

*Votre réponse :*


## 3. LightGBM avec `RandomizedSearchCV`

LightGBM = implémentation rapide de **gradient boosting**. Idée : construire les arbres **séquentiellement**, chacun corrigeant les erreurs des précédents. C'est le modèle **le plus performant** sur la majorité des données tabulaires.

On utilise `RandomizedSearchCV` plutôt que `GridSearchCV` : avec ~6 hyperparamètres importants, une grille exhaustive serait beaucoup trop coûteuse. On tire **20 combinaisons aléatoires** et on garde la meilleure.

> ⚠️ Cette cellule prend ~1–3 min.


In [ ]:
param_grid = {
    "learning_rate":   [0.05, 0.1, 0.15],
    "n_estimators":    [100, 200, 400],
    "num_leaves":      [15, 31, 63],
    "max_depth":       [-1, 5, 8],
    "subsample":       [0.7, 0.85, 1.0],
    "colsample_bytree":[0.7, 0.85, 1.0],
    "min_child_samples":[20, 50, 100],
}

lgbm = lgb.LGBMClassifier(objective="binary", verbosity=-1, random_state=42)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    lgbm, param_grid, n_iter=20, scoring="roc_auc",
    cv=cv, random_state=42, n_jobs=-1, verbose=0,
)
search.fit(X_train, y_train)

print("Meilleurs hyperparamètres :")
for k, v in search.best_params_.items():
    print(f"  {k} = {v}")
print(f"\nAUC CV (train) : {search.best_score_:.4f}")


In [ ]:
lgbm_best = search.best_estimator_
p_lgbm_test = lgbm_best.predict_proba(X_test)[:, 1]
auc_lgbm = roc_auc_score(y_test, p_lgbm_test)
print(f"AUC LightGBM (test) : {auc_lgbm:.4f}")


### Question 3
- Quel est l'écart d'AUC entre **arbre seul**, **RF** et **LightGBM** ? À quoi est dû le gain d'un modèle à l'autre (variance, biais) ?
- L'AUC en **validation croisée** (train) et sur le **test** sont-elles proches ? Si oui, que peut-on conclure sur le risque de surapprentissage ?
- Combien d'**arbres séquentiels** votre LightGBM a-t-il finalement construits (`n_estimators`) ? Et combien de **feuilles par arbre** ?

*Votre réponse :*


## 4. Comparaison globale : ROC et lift

On compare maintenant les 4 modèles (baseline GLM, arbre, RF, LightGBM) sur le **même graphique ROC** et le **même graphique de lift**. La probabilité du GLM amélioré du TP 5 est aussi rechargée.


In [ ]:
# Charger les probas GLM
p_glm_test = pd.read_csv("proba_glm_test.csv").squeeze("columns").values
p_glm_amel_test = pd.read_csv("proba_glm_ameliore_test.csv").squeeze("columns").values

models = {
    "GLM baseline":   p_glm_test,
    "GLM amélioré":   p_glm_amel_test,
    "Arbre":          p_tree_test,
    "Random Forest":  p_rf_test,
    "LightGBM":       p_lgbm_test,
}

fig, ax = plt.subplots(figsize=(7, 7))
for name, p in models.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    auc = roc_auc_score(y_test, p)
    ax.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")
ax.plot([0, 1], [0, 1], "--", color="gray")
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title("Courbes ROC — tous modèles")
ax.legend()
plt.show()


In [ ]:
def lift_cum(y_true, y_proba, n=100):
    df = pd.DataFrame({"y": y_true.values, "p": y_proba}).sort_values("p", ascending=False).reset_index(drop=True)
    df["bin"] = pd.qcut(df.index, q=n, labels=False, duplicates="drop") + 1
    out = df.groupby("bin").agg(n=("y", "size"), n_bad=("y", "sum"))
    out["%_pop_cum"] = out["n"].cumsum() / out["n"].sum()
    out["%_bad_cum"] = out["n_bad"].cumsum() / out["n_bad"].sum()
    return out

fig, ax = plt.subplots(figsize=(7, 7))
for name, p in models.items():
    lc = lift_cum(y_test, p, n=100)
    ax.plot([0] + list(lc["%_pop_cum"]), [0] + list(lc["%_bad_cum"]), label=name)
ax.plot([0, 1], [0, 1], "--", color="gray", label="Aléatoire")
ax.set_xlabel("% cumulé du portefeuille (trié par risque décroissant)")
ax.set_ylabel("% cumulé de sinistres BAD capturés")
ax.set_title("Courbes de lift cumulées (Gain Charts)")
ax.legend()
plt.show()


In [ ]:
# Tableau récapitulatif : AUC, Gini, Lift décile 1
def lift_d1(y_true, y_proba):
    df = pd.DataFrame({"y": y_true.values, "p": y_proba}).sort_values("p", ascending=False).reset_index(drop=True)
    top = df.head(int(len(df) * 0.1))
    return top["y"].mean() / df["y"].mean()

summary = pd.DataFrame(
    {name: {"AUC": roc_auc_score(y_test, p),
            "Gini": 2 * roc_auc_score(y_test, p) - 1,
            "Lift décile 1": lift_d1(y_test, p)}
     for name, p in models.items()}
).T.round(4)
print(summary)
summary.to_csv("comparison_models.csv")


### Question 4 — interprétation
- Quel modèle gagne sur **l'AUC** ? Sur le **lift décile 1** ?
- Le LightGBM gagne-t-il sur les **deux** ? Si non, comment interpréter le désaccord ?
- En tarification, on insiste souvent sur le **lift haut de gamme** (les 10–20% les plus risqués). Ce critère désigne-t-il un vainqueur clair ?

*Votre réponse :*


## 4 bis. Diagnostic — vraie ou fausse victoire du LightGBM ?

L'écart d'AUC entre LightGBM (~0,85) et le GLM (~0,58) est **énorme**. Avant de crier victoire, un actuaire prudent se demande : *est-ce un vrai gain de performance, ou un artefact ?*

Le risque ici est très spécifique au preprocessing du TP 2 : après recodage `VehAge_num` (9 valeurs), `VehMaxSpeed_num` (10), one-hot et binaires, **toutes les variables sont discrètes**. Beaucoup de contrats partagent **exactement le même vecteur de features**. On parle de **profils**.

Si un modèle très flexible (LightGBM avec `num_leaves=63`, `max_depth=-1`, 400 arbres) **mémorise** le taux de BAD par profil, alors :
- sur un profil **déjà rencontré** en train, la prédiction est presque parfaite ;
- sur un profil **inédit**, le modèle est aveugle.

Les deux cellules ci-dessous **comptent les profils** puis **séparent l'AUC test** selon que le profil a été vu ou non en entraînement.



In [ ]:
# 1. Combien de profils uniques ?  Combien de profils test sont déjà vus en train ?
key_train = X_train.apply(tuple, axis=1)
key_test  = X_test.apply(tuple, axis=1)

n_uniq_train = key_train.nunique()
n_uniq_test  = key_test.nunique()
seen = key_test.isin(set(key_train))

print(f"Train : {len(X_train):>5d} lignes  →  {n_uniq_train:>4d} profils uniques")
print(f"Test  : {len(X_test):>5d} lignes  →  {n_uniq_test:>4d} profils uniques")
print(f"\nLignes test dont le profil EXACT est déjà présent en train : "
      f"{seen.sum()} / {len(X_test)} = {seen.mean():.1%}")



In [ ]:
# 2. AUC test du LightGBM et du GLM, séparée selon que le profil est déjà vu ou non.
# (On (re)charge les probas pour rendre la cellule autonome ; elles ont été
# sauvegardées par les sections 3 (LightGBM) et par le TP 3 (GLM).)
p_lgbm_test = pd.read_csv("proba_lgbm_test.csv").squeeze("columns").values
p_glm_test  = pd.read_csv("proba_glm_test.csv").squeeze("columns").values

mask_seen   = seen.values
mask_unseen = ~mask_seen

def _auc_safe(y, p, m):
    if m.sum() < 30 or y[m].sum() < 5:
        return None
    return roc_auc_score(y[m], p[m])

rows = []
for name, p in [("LightGBM", p_lgbm_test), ("GLM baseline", p_glm_test)]:
    rows.append({
        "modèle":            name,
        "AUC global":        roc_auc_score(y_test, p),
        "AUC profils vus":   _auc_safe(y_test.values, p, mask_seen),
        "AUC profils nouveaux": _auc_safe(y_test.values, p, mask_unseen),
        "n vus":             int(mask_seen.sum()),
        "n nouveaux":        int(mask_unseen.sum()),
    })
diag = pd.DataFrame(rows).set_index("modèle").round(3)
print(diag)

# 3. Vérification : la proba LightGBM est-elle déterministe sur le profil ?
disp = pd.DataFrame({"key": key_test, "p": p_lgbm_test}).groupby("key")["p"].std()
print(f"\nDispersion (écart-type) de la proba LightGBM au sein d'un même profil test : "
      f"médiane = {disp.median():.6f}  (≈ 0 ⇒ le modèle prédit la MÊME valeur pour tous "
      f"les contrats d'un même profil — donc il a mémorisé un taux par profil)")



### Question 4 bis — diagnostic critique

- **Combien de profils uniques** y a-t-il en train ? En test ? Quelle proportion des lignes test correspond à un profil **déjà vu** en train ?
- Comparez l'AUC du LightGBM **sur les profils déjà vus** vs **sur les profils nouveaux**. Que constatez-vous ? Et pour le GLM, l'écart est-il du même ordre ?
- Que signifie le fait que la **dispersion de la proba LightGBM au sein d'un même profil test est ≈ 0** ?
- Conclusion : le LightGBM apprend-il à *prédire le risque*, ou à *retrouver le taux empirique d'un profil déjà rencontré* ? Qu'est-ce que cela implique pour la **généralisation** à un portefeuille en production, où apparaîtront en permanence de nouveaux profils ?

> 💡 **Leçon** : un `train_test_split` aléatoire ne suffit pas à valider un modèle quand les features sont fortement discrètes et que la cardinalité totale des profils (≪ taille du portefeuille) est faible. La **CV K-fold** souffre du même biais. Solutions à connaître :
> - **out-of-time validation** (split temporel) — le test ne contient que des contrats postérieurs au train, donc beaucoup de profils inédits ;
> - **régularisation forte** sur LightGBM (`num_leaves` faible, `min_child_samples` élevé) pour empêcher la mémorisation ;
> - **monitoring de production** sur le taux de profils inédits (TP 10).

*Votre réponse :*



## 5. Importance des variables — LightGBM vs GLM

Le LightGBM fournit deux mesures :
- **`split`** : combien de fois la variable est utilisée comme variable de séparation.
- **`gain`** : gain total en log-vraisemblance apporté par la variable.

Le **gain** est plus parlant. On le compare aux **odds ratios** du GLM.


In [ ]:
imp = pd.DataFrame({
    "feature": X_train.columns,
    "gain":  lgbm_best.booster_.feature_importance(importance_type="gain"),
    "split": lgbm_best.booster_.feature_importance(importance_type="split"),
}).sort_values("gain", ascending=False)

top20 = imp.head(20)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top20["feature"][::-1], top20["gain"][::-1], color="steelblue", edgecolor="k")
ax.set_xlabel("Gain total (LightGBM)")
ax.set_title("Top 20 variables — LightGBM")
plt.tight_layout()
plt.show()
print(top20.to_string(index=False))


### Question 5
- Comparez le **top 5** des variables LightGBM (par gain) avec le **top 5** des coefficients GLM les plus significatifs (TP 3). Y a-t-il un fort recouvrement ?
- Si une variable a un **gros gain LightGBM** mais un **coefficient GLM non significatif**, quelle hypothèse cela suggère-t-il ?
- Citez une **limite** de l'importance par gain comme outil d'interprétation (pensez aux variables corrélées).

*Votre réponse :*


## 6. Synthèse et préparation du TP 8

### Question 6 — synthèse
- Sur ce dataset, le LightGBM **bat-il vraiment** le GLM amélioré ? De **combien** (en points de Gini) ? Le gain vous paraît-il significatif compte tenu :
  - du coût de **maintenance** supplémentaire (centaines d'arbres vs ~50 coefficients),
  - du coût en **interprétabilité** vis-à-vis du régulateur ACPR,
  - de la complexité de **monitoring** en production (TP 10) ?
- Si la direction vous demande **un seul** modèle pour la prochaine campagne tarifaire, lequel choisissez-vous ? Justifiez.

*Votre réponse :*


In [ ]:
# Sauvegarde des probas LightGBM pour le TP 8 (interprétabilité SHAP)
pd.DataFrame({"proba": p_lgbm_test}).to_csv("proba_lgbm_test.csv", index=False)
pd.DataFrame({"proba": lgbm_best.predict_proba(X_train)[:, 1]}).to_csv("proba_lgbm_train.csv", index=False)

# Sauvegarde du modèle lui-même
import pickle
with open("model_lgbm.pkl", "wb") as f:
    pickle.dump(lgbm_best, f)
print("Sauvegardes : proba_lgbm_train.csv, proba_lgbm_test.csv, model_lgbm.pkl")


---
**Prochain TP : TP 8 — Interprétabilité avec SHAP.**

Comment expliquer une **prédiction individuelle** d'un modèle complexe comme LightGBM ? C'est crucial pour le métier, le client et le régulateur.
